# Gold Set PII Evaluation — Ablation & Analysis

This notebook loads evaluation results from the gold-set pipeline and produces:
1. Ablation comparison table across configs
2. Per-type recall heatmap
3. Per-language breakdown
4. Error analysis (false positives, false negatives)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "SimHei", "Arial Unicode MS"]

RUNS_DIR = Path("../data/runs/pii_eval")

## 1. Load Evaluation Results

In [ ]:
# Define ablation configs to compare
ABLATION_CONFIGS = {
    "baseline_v1": "pii_eval_gold_baseline_v1",
    # Add more run_ids as you complete ablation runs:
    # "rules_v2": "pii_eval_gold_rules_v2",
    # "rules_v2_ner_ft": "pii_eval_gold_rules_v2_ner_ft",
    # "full_ensemble": "pii_eval_gold_full_ensemble",
}

results = {}
for config_name, run_id in ABLATION_CONFIGS.items():
    summary_path = RUNS_DIR / run_id / "summary.json"
    if summary_path.exists():
        with summary_path.open() as f:
            results[config_name] = json.load(f)
        print(f"Loaded {config_name}: {summary_path}")
    else:
        print(f"Missing {config_name}: {summary_path}")

print(f"\nLoaded {len(results)} configs")

## 2. Overall Ablation Comparison

In [ ]:
rows = []
for config_name, data in results.items():
    o = data["overall"]
    rows.append({
        "Config": config_name,
        "TP": o["tp"],
        "FP": o["fp"],
        "FN": o["fn"],
        "Precision": o["precision"],
        "Recall": o["recall"],
        "F1": o["f1"],
        "F2": o.get("f2", 0),
    })

df_overall = pd.DataFrame(rows)
df_overall = df_overall.set_index("Config")
df_overall.style.format({
    "Precision": "{:.3f}",
    "Recall": "{:.3f}",
    "F1": "{:.3f}",
    "F2": "{:.3f}",
}).highlight_max(subset=["Recall", "F1", "F2"], color="lightgreen")

## 3. Per-Type Recall Heatmap

In [ ]:
pii_types = ["NAME", "PHONE", "EMAIL", "ADDRESS", "ID"]

type_data = {}
for config_name, data in results.items():
    per_type = data.get("per_type", {})
    type_data[config_name] = {
        t: per_type.get(t, {}).get("recall", 0.0) for t in pii_types
    }

df_type_recall = pd.DataFrame(type_data).T
df_type_recall.columns = pii_types

fig, ax = plt.subplots(figsize=(10, max(3, len(results) * 0.8)))
im = ax.imshow(df_type_recall.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(pii_types)))
ax.set_xticklabels(pii_types)
ax.set_yticks(range(len(df_type_recall)))
ax.set_yticklabels(df_type_recall.index)

for i in range(len(df_type_recall)):
    for j in range(len(pii_types)):
        val = df_type_recall.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="black" if val > 0.4 else "white", fontsize=11)

ax.set_title("Per-Type Recall by Configuration")
fig.colorbar(im, ax=ax, label="Recall")
plt.tight_layout()
plt.show()

## 4. Per-Language Breakdown

In [ ]:
for config_name, data in results.items():
    print(f"\n=== {config_name} ===")
    per_lang = data.get("per_lang", {})
    lang_rows = []
    for lang, metrics in per_lang.items():
        lang_rows.append({
            "Language": lang,
            "TP": metrics["tp"],
            "FP": metrics["fp"],
            "FN": metrics["fn"],
            "P": metrics["precision"],
            "R": metrics["recall"],
            "F1": metrics["f1"],
            "F2": metrics.get("f2", 0),
        })
    if lang_rows:
        display(pd.DataFrame(lang_rows).set_index("Language"))

## 5. Error Analysis

In [ ]:
# Load error records from the most recent/best config
best_config = list(results.keys())[-1] if results else None
if best_config:
    run_id = ABLATION_CONFIGS[best_config]
    errors_path = RUNS_DIR / run_id / "errors.jsonl"
    if errors_path.exists():
        errors = []
        with errors_path.open() as f:
            for line in f:
                if line.strip():
                    errors.append(json.loads(line))

        # False Negatives by type
        fn_by_type = {}
        fp_by_type = {}
        for err in errors:
            for fn in err.get("false_negatives", []):
                t = fn.get("type", "?")
                fn_by_type.setdefault(t, []).append(fn.get("text", ""))
            for fp in err.get("false_positives", []):
                t = fp.get("type", "?")
                fp_by_type.setdefault(t, []).append(fp.get("text", ""))

        print(f"Config: {best_config}")
        print(f"Records with errors: {len(errors)}")
        print(f"\n--- False Negatives (missed PII) ---")
        for t, texts in sorted(fn_by_type.items()):
            print(f"  {t} ({len(texts)}): {texts[:5]}")
        print(f"\n--- False Positives (spurious detections) ---")
        for t, texts in sorted(fp_by_type.items()):
            print(f"  {t} ({len(texts)}): {texts[:5]}")
    else:
        print(f"No errors file found at {errors_path}")
else:
    print("No results loaded")

## 6. Per-Record Detail

In [ ]:
if best_config:
    run_id = ABLATION_CONFIGS[best_config]
    per_record_path = RUNS_DIR / run_id / "per_record.jsonl"
    if per_record_path.exists():
        records = []
        with per_record_path.open() as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
        df_rec = pd.DataFrame(records)
        print(f"Per-record summary ({len(df_rec)} records):")
        display(df_rec.sort_values("f1").head(20))